In [1]:
# ── 패키지 설치 ──────────────────────────────────────────────
# !pip install langchain langchain-community langchain-openai langchain-text-splitters langgraph --upgrade
# !pip install google-genai  # Gemini 이미지 생성 패키지


In [2]:
!pip install google-genai


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\pc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# ── 환경 변수 로드 ──────────────────────────────────────────
# GEMINI_API_KEY도 .env에 추가 필요: GEMINI_API_KEY=발급받은키
from dotenv import load_dotenv

load_dotenv(r"C:\Users\pc\Desktop\src\.env")


True

In [4]:
# ── LangSmith 추적 설정 ──────────────────────────────────────
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

logging.langsmith("CH15-Agent-Projects")


LangSmith 추적을 시작합니다.
[프로젝트명]
CH15-Agent-Projects


In [5]:
# ── Tavily 웹 검색 도구 ───────────────────────────────────────
# 에이전트가 최신 웹 정보를 검색할 때 사용합니다.
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(k=6)


C:\Users\pc\AppData\Local\Temp\ipykernel_47368\3932800272.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(k=6)


다음은 retriever 를 생성하고, 생성한 retriever 를 기반으로 도구를 생성합니다.

먼저, 문서를 로드하고, 분할한 뒤 retriever 를 생성합니다.

In [6]:
# ── PDF 로드 → 청크 분할 → 벡터 스토어 → Retriever ──────────
# langchain 1.x 기준 올바른 import 경로:
#   langchain_text_splitters, langchain_community.vectorstores/document_loaders
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_docs = loader.load_and_split(text_splitter)
vector = FAISS.from_documents(split_docs, OpenAIEmbeddings())
retriever = vector.as_retriever()


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# ── Retriever를 에이전트 도구로 변환 ──────────────────────────
# document_prompt: 검색 결과에 페이지 번호, 파일명을 포함시켜
#                  에이전트가 출처를 명시할 수 있게 합니다.
from langchain_core.tools.retriever import create_retriever_tool
from langchain_core.prompts import PromptTemplate

document_prompt = PromptTemplate.from_template(
    "<document><content>{page_content}</content><page>{page}</page><filename>{source}</filename></document>"
)

retriever_tool = create_retriever_tool(
    retriever,
    name="pdf_search",
    description="use this tool to search for information in the PDF file",
    document_prompt=document_prompt,
)


In [8]:
# ── Retriever 도구 테스트 ─────────────────────────────────────
print(retriever_tool.invoke("삼성전자가 개발한 `가우디아 AI` 에 대한 내용을 찾아주세요."))


<document><content>SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 
처리를 지원
∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 
사내 소프트웨어 개발에 최적화
∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며,</content><page>12

In [9]:
# ── Pollinations.ai 이미지 생성 도구 (URL 반환 방식) ────────
# 이미지를 직접 다운로드하지 않고 URL만 반환합니다.
# 에이전트가 이 URL을 markdown ![이미지](url) 형식으로 보고서에 삽입합니다.
# Pollinations.ai는 URL 접근 시 즉석에서 이미지를 생성합니다.
from langchain.tools import tool
from urllib.parse import quote


@tool
def image_gen_tool(query: str) -> str:
    """use this tool to generate image from text"""
    url = f"https://image.pollinations.ai/prompt/{quote(query)}?width=1024&height=1024&nologo=true"
    return f"이미지 URL 생성 완료.\n마크다운: ![{query}]({url})"


### 파일 관리 도구

**파일 관리 도구들**

- `CopyFileTool`: 파일 복사
  
- `DeleteFileTool`: 파일 삭제

- `FileSearchTool`: 파일 검색

- `MoveFileTool`: 파일 이동

- `ReadFileTool`: 파일 읽기

- `WriteFileTool`: 파일 쓰기

- `ListDirectoryTool`: 디렉토리 목록 조회

In [10]:
# ── 파일 관리 도구 설정 ───────────────────────────────────────
# write_file, read_file, list_directory 3가지만 선택합니다.
from langchain_community.agent_toolkits import FileManagementToolkit

working_directory = "tmp"

file_tools = FileManagementToolkit(
    root_dir=str(working_directory),
    selected_tools=["write_file", "read_file", "list_directory"],
).get_tools()

file_tools


[WriteFileTool(root_dir='tmp'),
 ReadFileTool(root_dir='tmp'),
 ListDirectoryTool(root_dir='tmp')]

In [11]:
# ── 전체 도구 목록 구성 ───────────────────────────────────────
# 에이전트가 사용할 4가지 도구:
#   file_tools: 파일 쓰기/읽기/목록
#   retriever_tool: PDF 검색
#   search: Tavily 웹 검색
#   image_gen_tool: Gemini 이미지 생성
tools = file_tools + [
    retriever_tool,
    search,
    image_gen_tool,
]

tools


[WriteFileTool(root_dir='tmp'),
 ReadFileTool(root_dir='tmp'),
 ListDirectoryTool(root_dir='tmp'),
 StructuredTool(name='pdf_search', description='use this tool to search for information in the PDF file', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000182CC77BF60>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000182CF2A8040>),
 TavilySearchResults(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'))),
 StructuredTool(name='image_gen_tool', description='use this tool to generate image from text', args_schema=<class 'langchain_core.utils.pydantic.image_gen_tool'>, func=<function image_gen_tool at 0x00000182CF2F39C0>)]

## Agent 생성

In [12]:
# ── 에이전트 생성 (LangGraph 방식) ───────────────────────────
# create_react_agent + MemorySaver로 thread_id 기반 대화 히스토리 관리
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_teddynote.messages import AgentStreamParser

system_prompt = (
    "You are a helpful assistant. "
    "You are a professional researcher. "
    "You can use the pdf_search tool to search for information in the PDF file. "
    "You can find further information by using search tool. "
    "You can use image generation tool to generate image from text. "
    "Finally, you can use file management tool to save your research result into files."
)

llm = ChatOpenAI(model="gpt-4o-mini")
memory = MemorySaver()
agent_executor = create_react_agent(
    llm,
    tools,
    checkpointer=memory,
    prompt=system_prompt,
)
agent_with_chat_history = agent_executor
agent_stream_parser = AgentStreamParser()
print("에이전트 준비 완료")


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


에이전트 준비 완료


C:\Users\pc\AppData\Local\Temp\ipykernel_47368\3193827814.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [13]:
# ── [1단계] PDF 검색 후 report.md 작성 ──────────────────────
# pdf_search 도구로 내용을 찾고 bullet point 형식으로 report.md에 저장합니다.
result = agent_with_chat_history.stream(
    {
        "messages": [HumanMessage(content=
            "삼성전자가 개발한 `가우디아 AI` 에 관련된 내용을 상세하게 PDF 문서에서 찾아서 bullet point로 작성해 주세요. "
            "한글로 작성해주세요."
            "작성이후에는 `report.md` 파일에 깔끔하게 저장하여 파일로 저장해 주세요. \n\n"
            "#작성규칙: \n"
            "1. markdown header 2 크기로 섹션을 나눠서 작성하세요. \n"
            "2. 출처는 PDF 파일의 페이지 번호, 파일명을 명시하세요(예시: page 10, filename.pdf). \n"
            "3. 내용은 bullet point로 작성하세요. \n"
            "4. 작성이 완료되면 파일을 `report.md` 에 저장하세요. \n"
            "5. 최종적으로 저장된 `report.md` 파일을 읽어서 보여줘 주세요. \n"
        )]
    },
    config={"configurable": {"thread_id": "abc200"}},
    stream_mode="values"
)

print("Agent 실행 결과:")
for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


Agent 실행 결과:
삼성전자가 개발한 `가우디아 AI` 에 관련된 내용을 상세하게 PDF 문서에서 찾아서 bullet point로 작성해 주세요. 한글로 작성해주세요.작성이후에는 `report.md` 파일에 깔끔하게 저장하여 파일로 저장해 주세요. 

#작성규칙: 
1. markdown header 2 크기로 섹션을 나눠서 작성하세요. 
2. 출처는 PDF 파일의 페이지 번호, 파일명을 명시하세요(예시: page 10, filename.pdf). 
3. 내용은 bullet point로 작성하세요. 
4. 작성이 완료되면 파일을 `report.md` 에 저장하세요. 
5. 최종적으로 저장된 `report.md` 파일을 읽어서 보여줘 주세요. 



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


<document><content>저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 
2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기기 및 구글 
어시스턴트를 적용한 구글 픽셀(Pixel)과 경쟁할 것으로 예상
☞ 출처 : 삼성전자, ‘삼성 AI 포럼’서 자체 개발 생성형 AI ‘삼성 가우스’ 공개, 2023.11.08.
삼성전자, ‘삼성 개발자 콘퍼런스 코리아 2023’ 개최, 2023.11.14.
TechRepublic, Samsung Gauss: Samsung Research Reveals Generative AI, 2023.11.08.</content><page>12</page><filename>data/SPRI_AI_Brief_2023년12월호_F.pdf</filename></document>

<document><content>SPRi AI Brief |  
2023-12월호
4
미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각
n 미국 캘리포니아 북부지방법원은 미드저니, 스태빌리티AI, 디비언트아트를 대상으로 예술가 
3인이 제기한 저작권 침해 소송을 기각
n 법원은 기각 이유로 고소장에 제시된 상당수 작품이 저작권청에 등록되지 않았으며, AI로 
생성된 이미지와 특정 작품 간 유사성을 입증하기 어렵다는 점을 제시 
KEY Contents
£ 예술가들의 AI 저작권 침해 소송, 저작권 미등록과 증거불충분으로 기각
n 미국 캘리포니아 북부지방법원의 윌리엄 오릭(William Orrick) 판사는 2023년 10월 30일 미드저니
(Midjourney), 스태빌리티AI(Stability AI), 디비언트아트(DeviantArt)에 제기된 저작권 침해 소송을 기각 
∙2023년 1월 예술가 사라 앤더슨(Sarah Anderson), 캘리 맥커넌(Kelly McKernan), 칼라 
오르티즈(Karla 

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


File written successfully to report.md.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 삼성 가우디아 AI 개요

- 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 AI 모델인 ‘삼성 가우스’를 공개함.  
  출처: page 10, SPRI_AI_Brief_2023년12월호_F.pdf

- 삼성 가우스는 다양한 제품에 단계적으로 탑재될 계획이며, 온디바이스 작동이 가능하여 외부로 사용자 정보가 유출될 위험이 없다는 장점이 있음.  
  출처: page 10, SPRI_AI_Brief_2023년12월호_F.pdf

- 삼성 가우스는 다음의 3개 주요 모델로 구성됨:  
  - **언어 모델**: 메일 작성, 문서 요약, 번역 등의 지원. 클라우드와 온디바이스에서 다양한 모델로 구성됨.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf
  - **코드 모델**: AI 코딩 어시스턴트인 ‘코드아이(code.i)’가 대화형 인터페이스로 제공, 사내 소프트웨어 개발에 최적화되어 있음.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf
  - **이미지 모델**: 창의적인 이미지를 생성하고 기존 이미지를 원하는 형태로 바꿀 수 있으며 저해상도 이미지의 고해상도 전환도 지원.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf

- 삼성전자는 2023년 11월 8일에 열린 ‘삼성 AI 포럼 2023’ 행사에서 삼성 가우스를 최초로 공개함.  
  출처: page 10, SPRI_AI_Brief_2023년12월호_F.pdf

- IT 전문지 테크리퍼블릭에서는 2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기기 및 구글 어시스턴트를 적용한 구글 픽셀(Pixel)과 경쟁할 것으로 예상하고 있음.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


다음은 `report.md` 파일에 저장된 내용입니다:

## 삼성 가우디아 AI 개요

- 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 AI 모델인 ‘삼성 가우스’를 공개함.  
  출처: page 10, SPRI_AI_Brief_2023년12월호_F.pdf

- 삼성 가우스는 다양한 제품에 단계적으로 탑재될 계획이며, 온디바이스 작동이 가능하여 외부로 사용자 정보가 유출될 위험이 없다는 장점이 있음.  
  출처: page 10, SPRI_AI_Brief_2023년12월호_F.pdf

- 삼성 가우스는 다음의 3개 주요 모델로 구성됨:  
  - **언어 모델**: 메일 작성, 문서 요약, 번역 등의 지원. 클라우드와 온디바이스에서 다양한 모델로 구성됨.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf
  - **코드 모델**: AI 코딩 어시스턴트인 ‘코드아이(code.i)’가 대화형 인터페이스로 제공, 사내 소프트웨어 개발에 최적화되어 있음.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf
  - **이미지 모델**: 창의적인 이미지를 생성하고 기존 이미지를 원하는 형태로 바꿀 수 있으며 저해상도 이미지의 고해상도 전환도 지원.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf

- 삼성전자는 2023년 11월 8일에 열린 ‘삼성 AI 포럼 2023’ 행사에서 삼성 가우스를 최초로 공개함.  
  출처: page 10, SPRI_AI_Brief_2023년12월호_F.pdf

- IT 전문지 테크리퍼블릭에서는 2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기기 및 구글 어시스턴트를 적용한 구글 픽셀(Pixel)과 경쟁할 것으로 예상하고 있음.  
  출처: page 12, SPRI_AI_Brief_2023년12월호_F.pdf


생성된 보고서 파일(`report.md`)의 내용을 확인하면 다음과 같이 출력됩니다.

![](./assets/report-agent-01.png)

다음으로는 웹 검색을 통해 보고서 파일을 업데이트 해 봅시다.

In [14]:
# ── [2단계] 웹 검색 후 report.md에 추가 ─────────────────────
# Tavily로 최신 웹 정보를 검색하고 기존 report.md에 내용을 추가합니다.
result = agent_with_chat_history.stream(
    {
        "messages": [HumanMessage(content=
            "이번에는 웹 검색으로 삼성전자가 개발한 `가우디아 AI` 에 관련된 추가정보를 웹 검색해서, 검색한 내용을 요약해 주세요. "
            "한글로 작성해주세요."
            "작성이후에는 `report.md` 파일의 내용을 전체를 읽어서 보고, 웹 검색하여 찾은 내용을 요약한 뒤에 기존 파일 맨 뒤 부분에 추가해 주세요. \n\n"
            "#작성규칙: \n"
            "1. markdown header 2 크기로 섹션을 나눠서 작성하세요. \n"
            "2. 출처(url)는 명시하세요(예시: 출처: 네이버 뉴스기사). \n"
            "3. 내용은 웹검색 내용을 작성하세요. \n"
            "4. 작성이 완료되면 파일을 `report.md` 에 저장하세요. \n"
            "5. 최종적으로 저장된 `report.md` 파일을 읽어서 보여줘 주세요. \n"
        )]
    },
    config={"configurable": {"thread_id": "abc200"}},
    stream_mode="values"
)

print("Agent 실행 결과:")
for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


Agent 실행 결과:
이번에는 웹 검색으로 삼성전자가 개발한 `가우디아 AI` 에 관련된 추가정보를 웹 검색해서, 검색한 내용을 요약해 주세요. 한글로 작성해주세요.작성이후에는 `report.md` 파일의 내용을 전체를 읽어서 보고, 웹 검색하여 찾은 내용을 요약한 뒤에 기존 파일 맨 뒤 부분에 추가해 주세요. 

#작성규칙: 
1. markdown header 2 크기로 섹션을 나눠서 작성하세요. 
2. 출처(url)는 명시하세요(예시: 출처: 네이버 뉴스기사). 
3. 내용은 웹검색 내용을 작성하세요. 
4. 작성이 완료되면 파일을 `report.md` 에 저장하세요. 
5. 최종적으로 저장된 `report.md` 파일을 읽어서 보여줘 주세요. 



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[{"title": "삼성이 자체 개발한 생성형 AI ‘삼성 가우스’ 첫 공개", "url": "https://www.chosun.com/economy/tech_it/2023/11/08/IHCUCZUPXBCNDCCDYSA7NJFBHE/", "content": "조선경제테크\n\n# 삼성이 자체 개발한 생성형 AI '삼성 가우스' 첫 공개\n\n이해인 기자\n\n입력 2023.11.08. 10:02업데이트 2023.11.08. 10:04\n\n0\n\n삼성전자가 자체 개발한 생성형 AI 삼성 가우스 소개 포스터. /삼성전자\n\n삼성전자가 자체 개발한 생성형 AI 삼성 가우스 소개 포스터. /삼성전자\n\n삼성전자가 자체 개발한 생성형 인공지능(AI) ‘삼성 가우스’를 드디어 공개했다.\n\n8일 삼성전자는 서울 서초구 서울R&D캠퍼스에서 열린 ‘삼성 AI 포럼 2023′ 행사에서 삼성리서치가 개발한 생성형 AI 모델 ‘삼성 가우스’를 처음으로 공개했다. 삼성 가우스라는 명칭은 정규분포 이론을 정립한 수학자 칼 프리드리히 가우스로부터 영감을 얻어 붙여졌다. 생성형 AI의 무한한 가능성을 의미한다고 삼성은 밝혔다.\n\n삼성의 첫 자체 제작 생성 AI ‘삼성 가우스’는 머신 러닝 기술을 기반으로 개발됐다. 크게 텍스트를 생성하는 언어 모델과, 코드를 생성하는 코드 모델, 이미지를 생성하는 이미지 모델 등 3가지로 구성됐다.\n\n언어 모델은 문서 요약, 번역 등 업무를 더 쉽고 빠르게 처리할 수 있게 해준다. 코드 모델을 기반으로 개발된 AI 코딩 어시스턴트 ‘코드아이’는 사내 소프트웨어 개발에 활용될 예정이다. 개발자들이 쉽고 빠르게 코딩할 수 있도록 도와주고 코드 설명이나 테스트 케이스 생성 등의 서비스도 제공한다. [...] 이미지 모델은 달리 2(DALL·E), 미드저니, 스테이블 디퓨전 같은 생성형 AI 이미지 모델로 사진이나 그림 등 창의적인 이미지를 손쉽게 만들 수 있도록 돕는다. 기존 이미지를 원하는 대로 바꿀 수 있는 기능도 탑재돼 저해상도 이미지

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


File written successfully to report.md.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 웹 검색을 통한 삼성 가우디아 AI 추가 정보

- 삼성전자는 '삼성 가우스' AI 모델을 통해 사용자 경험을 다양한 기기에서 혁신하려고 함. 이 모델은 텍스트, 코드, 이미지 관련 다양한 비즈니스 응용 프로그램에서 메일 작성, 문서 요약, 번역 등을 지원함.  
출처: [조선일보](https://www.chosun.com/economy/tech_it/2023/11/08/IHCUCZUPXBCNDCCDYSA7NJFBHE/)

- 삼성 가우스는 클라우드와 온디바이스 환경 모두에서 사용자가 쉽게 이용할 수 있는 생성형 AI이며, 향후 갤럭시 S24와 같은 스마트폰에 적용될 예정임.  
출처: [ZDNet korea](https://zdnet.co.kr/view/?no=20231108081251)

- 삼성전자는 AI 기술 개발과 관련하여 사용자에게 발송되는 데이터 보안을 원칙으로 하고 있으며, 'AI 윤리 원칙'을 준수하고자 노력하고 있음.  
출처: [인더스트리뉴스](https://www.industrynews.co.kr/news/articleView.html?idxno=51475) 

- 생성형 AI 기술을 활용한 온디바이스 제품은 개인 정보 전송 없이도 기기 제어, 문장 요약, 문법 교정 등의 작업을 용이하게 처리할 수 있도록 설계됨.  
출처: [삼성 뉴스룸](https://news.samsung.com/kr/%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90-%EC%82%BC%EC%84%B1-ai-%ED%8F%AC%EB%9F%BC%EC%84%9C-%EC%9E%90%EC%B2%B4-%EA%B0%9C%EB%B0%9C-%EC%83%9D%EC%84%B1%ED%98%95-ai-%EC%82%BC%EC%84%B1-%EA%B0%80) 

- 삼성 가우스는 머신 러닝 기술에 기반하여 다양한 데이터셋을 활용하며, AI의 성능을 높이기 위한 최적의 학습 파라미터와 안정성을 추구하는 기술적 노력을 기울이고 있음.  
출처: [나무위키](https://namu

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


다음은 `report.md` 파일에 추가된 내용입니다:

## 웹 검색을 통한 삼성 가우디아 AI 추가 정보

- 삼성전자는 '삼성 가우스' AI 모델을 통해 사용자 경험을 다양한 기기에서 혁신하려고 함. 이 모델은 텍스트, 코드, 이미지 관련 다양한 비즈니스 응용 프로그램에서 메일 작성, 문서 요약, 번역 등을 지원함.  
  출처: [조선일보](https://www.chosun.com/economy/tech_it/2023/11/08/IHCUCZUPXBCNDCCDYSA7NJFBHE/)

- 삼성 가우스는 클라우드와 온디바이스 환경 모두에서 사용자가 쉽게 이용할 수 있는 생성형 AI이며, 향후 갤럭시 S24와 같은 스마트폰에 적용될予定임.  
  출처: [ZDNet korea](https://zdnet.co.kr/view/?no=20231108081251)

- 삼성전자는 AI 기술 개발과 관련하여 사용자에게 발송되는 데이터 보안을 원칙으로 하고 있으며, 'AI 윤리 원칙'을 준수하고자 노력하고 있음.  
  출처: [인더스트리뉴스](https://www.industrynews.co.kr/news/articleView.html?idxno=51475) 

- 생성형 AI 기술을 활용한 온디바이스 제품은 개인 정보 전송 없이도 기기 제어, 문장 요약, 문법 교정 등의 작업을 용이하게 처리할 수 있도록 설계됨.  
  출처: [삼성 뉴스룸](https://news.samsung.com/kr/%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90-%EC%82%BC%EC%84%B1-ai-%ED%8F%AC%EB%9F%BC%EC%84%9C-%EC%9E%90%EC%B2%B4-%EA%B0%9C%EB%B0%9C-%EC%83%9D%EC%84%B1%ED%98%95-ai-%EC%82%BC%EC%84%B1-%EA%B0%80) 

- 삼성 가우스는 머신 러닝 기술에 기반하여 다양한 데이터셋을 활용하며, AI의 성능을 높이기 위한 최적의 학습 파라미터와 안정성을 추구하는 기술적

업데이트된 보고서 파일(`report.md`)의 내용을 확인하면 다음과 같이 출력됩니다.

![](./assets/report-agent-02.png)

In [15]:
# ── [3단계] report.md 내용으로 report-2.md 구조화 작성 ──────
# 기존 report.md를 읽고 초록/핵심내용/참고문헌 구조로 report-2.md를 작성합니다.
result = agent_with_chat_history.stream(
    {
        "messages": [HumanMessage(content=
            "`report.md` 파일의 모든 내용을 활용하세요. "
            "항목별 내용을 요약하여, 전문적인 보고서 형식으로 작성하세요. "
            "전문적인 보고서는 총 3개의 섹션으로 구성되어야 합니다:\n"
            "1. 초록: 보고서의 abstract 을 300자 이내로 작성하세요.\n"
            "2. 핵심내용: 보고서의 핵심 내용을 작성하세요. 표가 필요할 때 markdown 테이블로 작성하여 추가하세요. "
            "3. 참고문헌: 보고서의 참고 문헌을 작성하세요. 출처(파일명, url 등)를 표시하세요."
            "최종적으로 작성된 보고서를 `report-2.md` 파일에 저장하세요."
        )]
    },
    config={"configurable": {"thread_id": "abc200"}},
    stream_mode="values"
)

print("Agent 실행 결과:")
for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


Agent 실행 결과:
`report.md` 파일의 모든 내용을 활용하세요. 항목별 내용을 요약하여, 전문적인 보고서 형식으로 작성하세요. 전문적인 보고서는 총 3개의 섹션으로 구성되어야 합니다:
1. 초록: 보고서의 abstract 을 300자 이내로 작성하세요.
2. 핵심내용: 보고서의 핵심 내용을 작성하세요. 표가 필요할 때 markdown 테이블로 작성하여 추가하세요. 3. 참고문헌: 보고서의 참고 문헌을 작성하세요. 출처(파일명, url 등)를 표시하세요.최종적으로 작성된 보고서를 `report-2.md` 파일에 저장하세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


File written successfully to report-2.md.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


# 삼성전자의 가우디아 AI 보고서

## 초록
이 보고서는 삼성전자가 개발한 생성형 AI 모델 ‘삼성 가우스’에 대한 상세한 개요와 함께 관련 웹 검색 정보를 바탕으로 한 추가 분석을 제시합니다. 삼성 가우스는 텍스트, 코드, 이미지 생성을 지원하는 온디바이스 AI로, 사용자 정보 보호와 다양한 제품에의 단계적 적용을 통해 사용자 경험 혁신을 목표로 하고 있습니다. 이를 통해 삼성전자는 AI 기술 발전과 안전성을 동시에 추구하고자 합니다.

## 핵심내용
### 1. 삼성 가우스 개요
- **발표 일자**: 2023년 11월 8일, 삼성 AI 포럼 2023에서 공개됨.
- **구성 모델**:  
  - **언어 모델**: 문서 요약, 번역, 메일 작성 지원.  
  - **코드 모델**: AI 코딩 어시스턴트 ‘코드아이’를 통한 소프트웨어 개발 지원.  
  - **이미지 모델**: 이미지 생성 및 수정 지원, 저해상도 이미지를 고해상도로 변환 가능.  

### 2. 기능 및 특징
| 모델                 | 기능                                                | 출처                                     |
|----------------------|-----------------------------------------------------|-----------------------------------------|
| 언어 모델            | 텍스트 생성 및 정보 요약, 번역 지원                | SPRI_AI_Brief_2023년12월호_F.pdf      |
| 코드 모델            | 코드 생성 지원, 대화형 인터페이스 제공            | SPRI_AI_Brief_2023년12월호_F.pdf      |
| 이미지 모델          | 창의적 이미지를 생성 및 기존 이미지 수정 지원      | SPRI_AI_Brief_20

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


다음은 `report-2.md` 파일에 저장된 전문적인 보고서 내용입니다:

# 삼성전자의 가우디아 AI 보고서

## 초록
이 보고서는 삼성전자가 개발한 생성형 AI 모델 ‘삼성 가우스’에 대한 상세한 개요와 함께 관련 웹 검색 정보를 바탕으로 한 추가 분석을 제시합니다. 삼성 가우스는 텍스트, 코드, 이미지 생성을 지원하는 온디바이스 AI로, 사용자 정보 보호와 다양한 제품에의 단계적 적용을 통해 사용자 경험 혁신을 목표로 하고 있습니다. 이를 통해 삼성전자는 AI 기술 발전과 안전성을 동시에 추구하고자 합니다.

## 핵심내용
### 1. 삼성 가우스 개요
- **발표 일자**: 2023년 11월 8일, 삼성 AI 포럼 2023에서 공개됨.
- **구성 모델**:  
  - **언어 모델**: 문서 요약, 번역, 메일 작성 지원.  
  - **코드 모델**: AI 코딩 어시스턴트 ‘코드아이’를 통한 소프트웨어 개발 지원.  
  - **이미지 모델**: 이미지 생성 및 수정 지원, 저해상도 이미지를 고해상도로 변환 가능.  

### 2. 기능 및 특징
| 모델                 | 기능                                                | 출처                                     |
|----------------------|-----------------------------------------------------|-----------------------------------------|
| 언어 모델            | 텍스트 생성 및 정보 요약, 번역 지원                | SPRI_AI_Brief_2023년12월호_F.pdf      |
| 코드 모델            | 코드 생성 지원, 대화형 인터페이스 제공            | SPRI_AI_Brief_2023년12월호_F.pdf      |
| 이미지 모델          | 창의적 이미지

![](./assets/report-agent-03.png)

In [16]:
# ── [4단계] Gemini로 이미지 생성 후 report-3.md에 추가 ───────
# image_gen_tool로 보고서 내용에 어울리는 이미지를 생성하고
# 이미지 경로를 markdown으로 report-3.md에 삽입합니다.
result = agent_with_chat_history.stream(
    {
        "messages": [HumanMessage(content=
            "`report-2.md` 파일의 모든 내용을 활용하세요. "
            "항목별 내용에 어울리는 이미지를 생성하세요. "
            "생성된 이미지의 url 을 markdown 형식으로 보고서의 각 섹션 이미지 추가하세요. "
            "최종적으로 작성된 보고서를 `report-3.md` 파일에 저장하세요."
        )]
    },
    config={"configurable": {"thread_id": "abc200"}},
    stream_mode="values"
)

print("Agent 실행 결과:")
for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


Agent 실행 결과:
`report-2.md` 파일의 모든 내용을 활용하세요. 항목별 내용에 어울리는 이미지를 생성하세요. 생성된 이미지의 url 을 markdown 형식으로 보고서의 각 섹션 이미지 추가하세요. 최종적으로 작성된 보고서를 `report-3.md` 파일에 저장하세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


이미지 URL 생성 완료.
마크다운: ![삼성 가우스의 기대 효과, 사용자 경험 혁신](https://image.pollinations.ai/prompt/%EC%82%BC%EC%84%B1%20%EA%B0%80%EC%9A%B0%EC%8A%A4%EC%9D%98%20%EA%B8%B0%EB%8C%80%20%ED%9A%A8%EA%B3%BC%2C%20%EC%82%AC%EC%9A%A9%EC%9E%90%20%EA%B2%BD%ED%97%98%20%ED%98%81%EC%8B%A0?width=1024&height=1024&nologo=true)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


File written successfully to report-3.md.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


# 삼성전자의 가우디아 AI 보고서

## 초록
이 보고서는 삼성전자가 개발한 생성형 AI 모델 ‘삼성 가우스’에 대한 상세한 개요와 함께 관련 웹 검색 정보를 바탕으로 한 추가 분석을 제시합니다. 삼성 가우스는 텍스트, 코드, 이미지 생성을 지원하는 온디바이스 AI로, 사용자 정보 보호와 다양한 제품에의 단계적 적용을 통해 사용자 경험 혁신을 목표로 하고 있습니다. 이를 통해 삼성전자는 AI 기술 발전과 안전성을 동시에 추구하고자 합니다.

![삼성전자의 가우디아 AI 개요, AI 모델 구성, 삼성 AI 포럼 2023](https://image.pollinations.ai/prompt/%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90%EC%9D%98%20%EA%B0%80%EC%9A%B0%EB%94%94%EC%95%84%20AI%20%EA%B0%9C%EC%9A%94%2C%20AI%20%EB%AA%A8%EB%8D%B8%20%EA%B5%AC%EC%84%B1%2C%20%EC%82%BC%EC%84%B1%20AI%20%ED%8F%AC%EB%9F%BC%202023?width=1024&height=1024&nologo=true)

## 핵심내용
### 1. 삼성 가우스 개요
- **발표 일자**: 2023년 11월 8일, 삼성 AI 포럼 2023에서 공개됨.
- **구성 모델**:  
  - **언어 모델**: 문서 요약, 번역, 메일 작성 지원.  
  - **코드 모델**: AI 코딩 어시스턴트 ‘코드아이’를 통한 소프트웨어 개발 지원.  
  - **이미지 모델**: 이미지 생성 및 수정 지원, 저해상도 이미지를 고해상도로 변환 가능.  

### 2. 기능 및 특징
| 모델                 | 기능                                                | 출처                                     |
|----------------------|---------------------

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


최종적으로 작성된 보고서 `report-3.md`의 내용은 다음과 같습니다:

# 삼성전자의 가우디아 AI 보고서

## 초록
이 보고서는 삼성전자가 개발한 생성형 AI 모델 ‘삼성 가우스’에 대한 상세한 개요와 함께 관련 웹 검색 정보를 바탕으로 한 추가 분석을 제시합니다. 삼성 가우스는 텍스트, 코드, 이미지 생성을 지원하는 온디바이스 AI로, 사용자 정보 보호와 다양한 제품에의 단계적 적용을 통해 사용자 경험 혁신을 목표로 하고 있습니다. 이를 통해 삼성전자는 AI 기술 발전과 안전성을 동시에 추구하고자 합니다.

![삼성전자의 가우디아 AI 개요, AI 모델 구성, 삼성 AI 포럼 2023](https://image.pollinations.ai/prompt/%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90%EC%9D%98%20%EA%B0%80%EC%9A%B0%EB%94%94%EC%95%84%20AI%20%EA%B0%9C%EC%9A%94%2C%20AI%20%EB%AA%A8%EB%8D%B8%20%EA%B5%AC%EC%84%B1%2C%20%EC%82%BC%EC%84%B1%20AI%20%ED%8F%AC%EB%9F%BC%202023?width=1024&height=1024&nologo=true)

## 핵심내용
### 1. 삼성 가우스 개요
- **발표 일자**: 2023년 11월 8일, 삼성 AI 포럼 2023에서 공개됨.
- **구성 모델**:  
  - **언어 모델**: 문서 요약, 번역, 메일 작성 지원.  
  - **코드 모델**: AI 코딩 어시스턴트 ‘코드아이’를 통한 소프트웨어 개발 지원.  
  - **이미지 모델**: 이미지 생성 및 수정 지원, 저해상도 이미지를 고해상도로 변환 가능.  

### 2. 기능 및 특징
| 모델                 | 기능                                                | 출처                                     |
|

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


![](./assets/report-agent-04.png)